In [ ]:
# Cell 1: Imports and environment (merged old + new)
import time, zlib, itertools, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, spearmanr, ks_2samp, norm
from scipy.optimize import brentq

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              balanced_accuracy_score, average_precision_score)

from statsmodels.stats.multitest import multipletests

SEED = 42
np.random.seed(SEED)

# ---- new constants (needed by baseline cells 19-30) ----
TOTAL_EPS  = 3.0
SEEDS      = [42, 123, 456, 789, 101]
TUNE_SEEDS = [1, 2, 3]
PAR_SEEDS  = [42, 123, 456, 789, 101, 202, 303, 404, 505, 606]

def mk_rng(seed, tag):
    """Reproducible, process-independent RNG per (seed, tag)."""
    return np.random.default_rng([int(seed), zlib.crc32(tag.encode())])

print("Environment ready. SEED=42, TOTAL_EPS=3.0, SEEDS=5.")

In [ ]:
# Cell 2 (Kaggle version): 10-dataset loader, old API preserved
import socket, time, os
from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np

socket.setdefaulttimeout(180)
CACHE_HOME = "/kaggle/working/openml_cache"
os.makedirs(CACHE_HOME, exist_ok=True)
SEED = 42

# ---- 10-dataset split (must match the new notebook) ----
PRIMARY = ["Banknote","BreastCancer","Ionosphere","MAGIC04","PimaDiabetes",
           "Sonar","Wine"]
SUPP    = ["Covertype","CreditG","HeartCleveland"]

def fetch_openml_safe(data_id, retries=6, base_delay=15):
    """Fetch OpenML dataset with retries, exponential backoff, and local caching."""
    last_err = None
    for attempt in range(retries):
        try:
            try:
                return fetch_openml(data_id=data_id, as_frame=True,
                                    parser="auto", data_home=CACHE_HOME)
            except TypeError:
                return fetch_openml(data_id=data_id, as_frame=True,
                                    data_home=CACHE_HOME)
        except Exception as e:
            last_err = e
            wait = base_delay * (2 ** attempt)
            print(f"  attempt {attempt+1}/{retries} failed ({type(e).__name__}); retry in {wait}s")
            time.sleep(wait)
    raise last_err

def check_openml_availability():
    print("Running pre-flight OpenML check with iris (ID 61)...")
    try:
        old_timeout = socket.getdefaulttimeout(); socket.setdefaulttimeout(20)
        _ = fetch_openml(data_id=61, as_frame=True, parser="auto", data_home=CACHE_HOME)
        socket.setdefaulttimeout(old_timeout)
        print("✅ OpenML is reachable (iris loaded)."); return True
    except TypeError:
        try:
            socket.setdefaulttimeout(20)
            _ = fetch_openml(data_id=61, as_frame=True, data_home=CACHE_HOME)
            socket.setdefaulttimeout(old_timeout)
            print("✅ OpenML is reachable (iris loaded)."); return True
        except Exception as e:
            socket.setdefaulttimeout(old_timeout)
            print(f"❌ OpenML check failed: {type(e).__name__}: {e}")
            return False
    except Exception as e:
        socket.setdefaulttimeout(old_timeout)
        print(f"❌ OpenML check failed: {type(e).__name__}: {e}")
        return False

openml_available = check_openml_availability()
OPENML_RETRIES    = 4 if openml_available else 2
OPENML_BASE_DELAY = 10 if openml_available else 5

def prepare_tabular(X, y, name="", max_rows=None, seed=SEED):
    X = X.copy()
    y = pd.Series(y).astype(str).str.strip()
    na_vals = ["?", "NA", "N/A", "na", "n/a", "null", "None", "", "nan"]
    y = y.replace(na_vals, np.nan)
    mask = y.notna()
    X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
    for col in X.columns:
        converted = pd.to_numeric(X[col], errors="coerce")
        if converted.notna().mean() > 0.90:
            X[col] = converted
        else:
            cat = X[col].astype(str).str.strip().replace(na_vals, np.nan)
            X[col] = pd.Categorical(cat).codes.astype(float)
            X.loc[cat.isna(), col] = np.nan
    X = X.dropna()
    y = y.loc[X.index].reset_index(drop=True)
    X = X.reset_index(drop=True)
    X = X.loc[:, X.nunique(dropna=True) > 1]
    y_enc = pd.Series(pd.factorize(y)[0], name="target")
    if len(X) < 100 or y_enc.nunique() < 2:
        raise ValueError(f"{name}: too few samples or single class after cleaning")
    if max_rows is not None and len(X) > max_rows:
        X, _, y_enc, _ = train_test_split(X, y_enc, train_size=max_rows/len(X),
                                          random_state=seed, stratify=y_enc)
        X, y_enc = X.reset_index(drop=True), y_enc.reset_index(drop=True)
    return X, y_enc

datasets = {}
CATEGORY_CARDINALITY = {}

def add_dataset(name, X, y, max_rows=None, category_cards=None):
    try:
        Xp, yp = prepare_tabular(X, y, name=name, max_rows=max_rows)
        datasets[name] = (Xp, yp)
        if category_cards:
            CATEGORY_CARDINALITY[name] = category_cards
        bal = yp.value_counts(normalize=True).round(3).to_dict()
        print(f"OK  {name:<18} n={len(Xp):<6} d={Xp.shape[1]:<3} "
              f"classes={yp.nunique()} balance={bal}")
    except Exception as e:
        print(f"SKIP {name:<18} reason: {e}")

bc = load_breast_cancer()
add_dataset("BreastCancer", pd.DataFrame(bc.data, columns=bc.feature_names), bc.target)

# ---- 9 OpenML datasets (BreastCancer from sklearn, above) ----
openml_sources = [
    (37,   "PimaDiabetes",   None),
    (59,   "Ionosphere",     None),
    (40,   "Sonar",          None),
    (187,  "Wine",           None),
    (1462, "Banknote",       None),
    (1120, "MAGIC04",        6000),
    (49,   "HeartCleveland", None),   # was "HeartHungarian" in the 22-dataset run
    (31,   "CreditG",        None),
    (150,  "Covertype",      6000),
]

for data_id, name, max_rows in openml_sources:
    try:
        d = fetch_openml_safe(data_id, retries=OPENML_RETRIES, base_delay=OPENML_BASE_DELAY)
        cards = {col: len(d.data[col].cat.categories)
                 for col in d.data.columns if hasattr(d.data[col], "cat")}
        add_dataset(name, d.data, d.target, max_rows=max_rows, category_cards=cards)
    except Exception as e:
        print(f"SKIP {name:<18} reason: {e}")
    time.sleep(2)

print(f"\n{len(datasets)} datasets loaded: {list(datasets.keys())}")

# ---- PUBLIC class counts (10 datasets only) ----
PUBLIC_N_CLASSES = {
    "BreastCancer": 2, "PimaDiabetes": 2, "HeartCleveland": 2, "CreditG": 2,
    "MAGIC04": 2, "Banknote": 2, "Covertype": 7,
    "Ionosphere": 2, "Sonar": 2, "Wine": 3,
}

classes_public = {
    ds_name: np.arange(PUBLIC_N_CLASSES[ds_name])
    for ds_name in datasets if ds_name in PUBLIC_N_CLASSES
}

# ---- Clusterability scoring ----
def dataset_clusterability(X_norm, y, seed, k=None):
    k = k or max(2, y.nunique())
    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X_norm)
    return silhouette_score(X_norm, km.labels_)

clusterability = {}
for ds_name, (X, y) in datasets.items():
    scaler = MinMaxScaler()
    X_n = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
    clusterability[ds_name] = dataset_clusterability(X_n, y, SEED)

print(pd.Series(clusterability).sort_values(ascending=False).to_string())


In [ ]:
# Cell 2b — Public feature bounds (10 datasets, old API: datasets + CATEGORY_CARDINALITY)
PUBLIC_BOUNDS = {
 "BreastCancer": {"mean radius":(6.981,28.11),"mean texture":(9.71,39.28),"mean perimeter":(43.79,188.5),
  "mean area":(143.5,2501.0),"mean smoothness":(0.05263,0.1634),"mean compactness":(0.01938,0.3454),
  "mean concavity":(0.0,0.4268),"mean concave points":(0.0,0.2012),"mean symmetry":(0.106,0.304),
  "mean fractal dimension":(0.04996,0.09744),"radius error":(0.1115,2.873),"texture error":(0.3602,4.885),
  "perimeter error":(0.757,21.98),"area error":(6.802,542.2),"smoothness error":(0.001713,0.03113),
  "compactness error":(0.002252,0.1354),"concavity error":(0.0,0.396),"concave points error":(0.0,0.05279),
  "symmetry error":(0.007882,0.07895),"fractal dimension error":(0.0008948,0.02984),
  "worst radius":(7.93,36.04),"worst texture":(12.02,49.54),"worst perimeter":(50.41,251.2),
  "worst area":(185.2,4254.0),"worst smoothness":(0.07117,0.2226),"worst compactness":(0.02729,1.058),
  "worst concavity":(0.0,1.252),"worst concave points":(0.0,0.291),"worst symmetry":(0.1565,0.6638),
  "worst fractal dimension":(0.05504,0.2075)},
 "PimaDiabetes": {"preg":(0,17),"plas":(0,199),"pres":(0,122),"skin":(0,99),"insu":(0,846),
  "mass":(0.0,67.1),"pedi":(0.078,2.42),"age":(21,81)},
 "MAGIC04": {"fLength:":(10.89,317.98),"fWidth:":(5.29,262.96),"fSize:":(0.4,1028.0),"fConc:":(0.05,0.99),
  "fConc1:":(0.0,0.94),"fAsym:":(-24.68,24.68),"fM3Long:":(-331.0,331.0),"fM3Trans:":(-331.0,331.0),
  "fAlpha:":(0.0,90.0),"fDist:":(20.0,500.0)},
 "Banknote": {"V1":(-7.0421,6.8248),"V2":(-13.7731,12.9516),"V3":(-5.2861,17.9274),"V4":(-8.5482,2.4495)},
 "Ionosphere": {**{f"a{str(i).zfill(2)}":(-1.0,1.0) for i in range(1,35)},
                **{f"V{i}":(-1.0,1.0) for i in range(1,35)}},
 "Sonar": {**{f"V{i}":(0.0,1.0) for i in range(1,61)},
           **{f"attribute_{i}":(0.0,1.0) for i in range(1,61)}},
 "Wine": {"Alcohol":(11.03,14.83),"Malic_acid":(0.74,5.80),"Ash":(1.36,3.23),"Alcalinity_of_ash":(10.6,30.0),
  "Magnesium":(70,162),"Total_phenols":(0.98,3.88),"Flavanoids":(0.34,5.08),"Nonflavanoid_phenols":(0.13,0.66),
  "Proanthocyanins":(0.41,3.58),"Color_intensity":(1.28,13.0),"Hue":(0.48,1.71),
  "OD280%2FOD315_of_diluted_wines":(1.27,4.00),"Proline":(278,1680)},
}

def public_bounds_for(name, X, widen=1.0):
    """PRIMARY: documented bounds (raises KeyError if uncovered).
       SUPP: observed extremes -> used ONLY for tuning/runtime, never for headline results.
       Matches the 10-dataset notebook's Cell 3 behavior."""
    if name in SUPP:
        lo = X.min().astype(float).values
        hi = X.max().astype(float).values
        hi = np.where(hi > lo, hi, lo + 1.0)
    else:
        doc   = PUBLIC_BOUNDS[name]
        cards = CATEGORY_CARDINALITY.get(name, {})
        lo = np.zeros(X.shape[1]); hi = np.ones(X.shape[1])
        for j, c in enumerate(X.columns):
            if   c in doc:   lo[j], hi[j] = doc[c]
            elif c in cards: lo[j], hi[j] = 0, cards[c] - 1
            else: raise KeyError(f"{name}: no public bound for column {c!r}")
    if widen != 1.0:
        mid = (lo + hi) / 2
        lo, hi = mid - widen*(mid - lo), mid + widen*(hi - mid)
    return lo, hi

# Coverage check — raises if any PRIMARY column is uncovered
for nm in PRIMARY:
    public_bounds_for(nm, datasets[nm][0])
print("bounds OK for all primary datasets")

In [ ]:
# Cell 3: DP-consistent granular-ball construction — core private mechanisms.
#
# Two private sub-mechanisms, both standard and citable:
#  (a) Private purity check -> Laplace mechanism on class counts.
#      Sensitivity = 1 (adding/removing one record changes a count by <= 1).
#  (b) Private split selection -> Exponential Mechanism over candidate
#      (feature, threshold) pairs, scored by the majority-count split score
#      (see Cell 3d), sensitivity = 1.
#
# Composition: fixed max_depth D is chosen BEFORE looking at data (public
# hyperparameter). At each depth, all nodes are disjoint partitions of the
# training set -> parallel composition -> one (eps_count + eps_split) charge
# per depth level, not per node. Total granulation cost = D * (eps_count + eps_split).
# =============================================================================
# NEIGHBOURING RELATION: ADD/REMOVE (one record added or removed).
# Under this definition, the sensitivity of any count is 1.
# All Laplace mechanisms in this notebook assume ADD/REMOVE.
# If you switch to SUBSTITUTE-ONE, you must change sensitivities to 2 where
# appropriate and re-derive budgets.
# =============================================================================
def laplace_mech(true_value, sensitivity, eps, rng):
    if eps <= 0:
        return true_value
    scale = sensitivity / eps
    return true_value + rng.laplace(0, scale)

def exponential_mechanism(candidates, scores, sensitivity, eps, rng):
    """candidates: list of items; scores: matching list of utility scores
    (higher = better). Returns one sampled candidate."""
    if eps <= 0 or len(candidates) == 0:
        return candidates[rng.integers(len(candidates))]
    scores = np.asarray(scores, dtype=float)
    logits = (eps * scores) / (2.0 * sensitivity)
    logits -= logits.max()
    probs = np.exp(logits)
    probs /= probs.sum()
    idx = rng.choice(len(candidates), p=probs)
    return candidates[idx]

print("Laplace mechanism + Exponential mechanism ready. "
      "The granular-ball builder itself is defined in Cell 3d "
      "(dp_generate_granular_balls_with_radius_fixed), the only version now kept.")



In [ ]:
# Cell 3c — shared helpers used by the granular-ball builder and by direct
# ball classification.
#
# No per-ball center/radius release: they are free public post-processing of the released split thresholds.
from scipy.spatial.distance import cdist

def dp_release_ball_counts(balls, eps_count_release, rng):
    """How many synthetic records to emit per ball. Sensitivity=1 (one record
    changes exactly one ball's true size by at most 1), pure-eps Laplace."""
    sizes = []
    for ball in balls:
        true_n = len(ball["indices"])
        noisy_n = laplace_mech(true_n, sensitivity=1.0, eps=eps_count_release, rng=rng)
        sizes.append(max(0, int(round(noisy_n))))
    return sizes


def dp_gbc_classify_predict(balls, X_te_scaled):
    """
    Classify directly from DP-released ball geometry -- no synthetic feature
    sampling. Post-processing of released (center, noisy_class_counts):
    zero additional budget.
    """
    centers = np.array([b["center"] for b in balls])
    X_te_arr = X_te_scaled.values if hasattr(X_te_scaled, "values") else np.asarray(X_te_scaled)
    dists = cdist(X_te_arr, centers)
    nearest = np.argmin(dists, axis=1)
    ball_preds = []
    for b in balls:
        counts = np.clip(b["noisy_class_counts"], 0, None)
        ball_preds.append(b["classes"][np.argmax(counts)] if counts.sum() > 0 else b["classes"][0])
    return np.array(ball_preds)[nearest]


In [ ]:
# Cell 3d (CORRECTED & FIXED): majority-count split score, sensitivity PROVABLY = 1.
# Gini gain (previous version) does not have a small, n-/balance-independent
# sensitivity bound -- see Fletcher & Islam 2019 survey noting Friedman &
# Schuster preferred the count-based "Max" operator specifically for its low,
# provable sensitivity over impurity-based scores: adding/removing one record
# changes exactly one class's count on exactly one side of a candidate split
# by exactly 1, so |Δscore| <= 1 always.
#
# NOTE: this cell now defines the ONLY granular-ball builder used anywhere in
# the notebook (dp_generate_granular_balls_with_radius_fixed). The earlier
# non-radius "fixed" variant was dead code -- nothing ever called it -- and
# has been removed, along with the stale-definition sanity check that existed
# only to guard against it.

# def majority_count_score(y_node, vals, t):
#     left_mask = vals <= t
#     y_left, y_right = y_node[left_mask], y_node[~left_mask]
#     if len(y_left) == 0 or len(y_right) == 0:
#         return 0.0
#     left_counts = np.bincount(y_left.astype(int))
#     right_counts = np.bincount(y_right.astype(int))
#     return float(left_counts.max() + right_counts.max())
def majority_count_score(y_node, vals, t):
    left_mask = vals <= t
    y_left, y_right = y_node[left_mask], y_node[~left_mask]
    left_contrib  = np.bincount(y_left.astype(int)).max()  if len(y_left)  > 0 else 0.0
    right_contrib = np.bincount(y_right.astype(int)).max() if len(y_right) > 0 else 0.0
    return float(left_contrib + right_contrib)
def dp_generate_granular_balls_with_radius_fixed(X, y, max_depth, eps_count_per_level,
                                                 eps_split_per_level, eps_radius_per_level,
                                                 eps_center_per_level=0.1,
                                                 min_size=10, n_candidate_thresholds=8,
                                                 rng=None, all_classes=None,
                                                 purity_tau=0.90):          # <-- NEW
    rng = rng or np.random.default_rng(SEED)
    X_arr = X.values if hasattr(X, "values") else np.asarray(X)
    y_arr = np.asarray(y)
    global_classes = all_classes if all_classes is not None else np.unique(y_arr)
    n_features = X_arr.shape[1]
    balls = []
    root_lo = np.zeros(n_features)
    root_hi = np.ones(n_features)
    queue = [(np.arange(len(X_arr)), 0, root_lo, root_hi)]

    while queue:
        indices, depth, box_lo, box_hi = queue.pop(0)
        X_node, y_node = X_arr[indices], y_arr[indices]

        counts_full = np.array([np.sum(y_node == c) for c in global_classes])
        noisy_counts = np.array([laplace_mech(c, sensitivity=1.0, eps=eps_count_per_level, rng=rng) for c in counts_full])
        noisy_counts = np.clip(noisy_counts, 0, None) + 1.0
        noisy_purity = noisy_counts.max() / max(noisy_counts.sum(), 1e-6)

        stop = (noisy_purity >= purity_tau) or (noisy_counts.sum() < min_size) or (depth >= max_depth)

        if stop:
            center = (box_lo + box_hi) / 2.0
            radius = np.linalg.norm(box_hi - box_lo) / 2.0
            balls.append({
                "indices": indices, "noisy_purity": noisy_purity,
                "center": center, "radius": radius,
                "box_lo": box_lo.copy(), "box_hi": box_hi.copy(),
                "n": len(indices), "is_boundary": noisy_purity < purity_tau,
                "classes": global_classes, "noisy_class_counts": np.clip(noisy_counts, 0, None)
            })
            continue

        candidates, scores = [], []
        for f in range(n_features):
            lo_f, hi_f = box_lo[f], box_hi[f]
            if hi_f - lo_f < 1e-9:
                continue
            vals = X_node[:, f]
            thresholds = np.linspace(lo_f, hi_f, n_candidate_thresholds + 2)[1:-1]
            for t in thresholds:
                candidates.append((f, t))
                scores.append(majority_count_score(y_node, vals, t))
        if not candidates:
            center = (box_lo + box_hi) / 2.0
            radius = np.linalg.norm(box_hi - box_lo) / 2.0
            balls.append({
                "indices": indices, "noisy_purity": noisy_purity,
                "center": center, "radius": radius,
                "box_lo": box_lo.copy(), "box_hi": box_hi.copy(),
                "n": len(indices), "is_boundary": noisy_purity < purity_tau,
                "classes": global_classes, "noisy_class_counts": np.clip(noisy_counts, 0, None)
            })
            continue

        chosen_f, chosen_t = exponential_mechanism(candidates, scores, 1.0, eps_split_per_level, rng)
        left_mask = X_node[:, chosen_f] <= chosen_t

        left_lo, left_hi = box_lo.copy(), box_hi.copy()
        left_hi[chosen_f] = np.clip(min(left_hi[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])
        right_lo, right_hi = box_lo.copy(), box_hi.copy()
        right_lo[chosen_f] = np.clip(max(right_lo[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])

        queue.append((indices[left_mask], depth + 1, left_lo, left_hi))
        queue.append((indices[~left_mask], depth + 1, right_lo, right_hi))

    return balls

# Single canonical name used by every other cell in the notebook.
dp_generate_granular_balls_with_radius = dp_generate_granular_balls_with_radius_fixed

print("FIX APPLIED: majority-count split score (per-side contribution, sensitivity=1), "
      "global class counts, public box radii, noised stopping, box-scoped thresholds, "
      "clipped child boxes. Dead/stale builder variants removed — one canonical implementation remains.")

In [ ]:
def dp_release_box_means(balls, eps_mean, rng, X_arr, clip_norm=1.0):
    """Release each ball's true mean with Gaussian noise on the clipped sum.
    Empty balls fall back to their (public) box midpoint."""
    means = []
    delta = 1e-5
    for b in balls:
        idx = b["indices"]
        n_B = len(idx)

        # --- Guard: empty ball -> public box midpoint (no data-dependent release) ---
        if n_B == 0:
            means.append((b["box_lo"] + b["box_hi"]) / 2.0)
            continue

        Xb = X_arr[idx]
        norms = np.linalg.norm(Xb, axis=1, keepdims=True)
        Xb_clip = Xb * np.minimum(1.0, clip_norm / np.maximum(norms, 1e-8))
        true_mean = Xb_clip.mean(axis=0)

        sens = clip_norm / n_B
        sigma = sens * np.sqrt(2 * np.log(1.25 / delta)) / max(eps_mean, 1e-6)
        noisy_mean = true_mean + rng.normal(0, sigma, size=true_mean.shape)
        noisy_mean = np.nan_to_num(noisy_mean, nan=0.5, posinf=1.0, neginf=0.0)
        means.append(np.clip(noisy_mean, 0.0, 1.0))
    return means


def dp_sample_truncated_gaussian(balls, sizes, means, rng, d,
                                  rel_std=0.25):
    """Box-truncated Gaussian sampling: center = released mean,
    std = rel_std * box width per coordinate, clipped back into the box."""
    X_synth, y_synth = [], []
    for ball, n_out, mean in zip(balls, sizes, means):
        if n_out <= 0:
            continue

        lo, hi = ball["box_lo"], ball["box_hi"]
        hi = np.maximum(hi, lo)
        width = np.maximum(hi - lo, 1e-6)
        std = rel_std * width

        # --- Guard: kill any residual NaN/Inf in the mean or std ---
        mean = np.nan_to_num(mean, nan=0.5, posinf=1.0, neginf=0.0)
        mean = np.clip(mean, lo, hi)
        std  = np.nan_to_num(std,  nan=0.25, posinf=1.0, neginf=0.0)

        X_new = rng.normal(loc=mean, scale=std, size=(int(n_out), d))
        X_new = np.clip(X_new, lo, hi)
        X_new = np.nan_to_num(X_new, nan=0.5, posinf=1.0, neginf=0.0)
        X_synth.append(X_new)

        classes = ball["classes"]
        noisy_counts = np.clip(
            np.asarray(ball["noisy_class_counts"], dtype=float), 0.0, None
        )
        if noisy_counts.sum() > 0:
            p = noisy_counts / noisy_counts.sum()
            y_new = rng.choice(classes, size=int(n_out), p=p)
        else:
            y_new = np.full(int(n_out), classes[0])
        y_synth.extend(y_new)

    if not X_synth:
        return np.empty((0, d)), np.empty(0, dtype=int)
    return np.vstack(X_synth), np.asarray(y_synth, dtype=int)

In [ ]:
# Cell 4 (FIXED): DP-GBC synthetic release.
# Replaces apply_region_noise (data-dependent Gaussian noise) with:
#   (1) DP-release a count per ball
#   (2) sample synthetic points from the ball's released statistics
# Sampling from released stats is pure post-processing, zero extra budget.
#
# FIX B: MinMax scaling is replaced by one-time DP-released feature bounds.
# Bounds are released once per dataset, outside the seed loop, and are public afterward.
# FIX C: Synthetic points are sampled uniformly from each ball's box, not an L2 sphere.
# FIX D: Label sampling uses 'classes' + 'noisy_class_counts' (not 'class_counts').
# FIX E: Global safeguard handles any number of classes (not just binary).
# NEW: sampling_mode parameter ("uniform" default, "gaussian" optional).


class PrivacyLedger:
    """Tracks spent (ε, δ) privacy budget using sequential and parallel composition."""
    def __init__(self):
        self.eps_spent = 0.0
        self.delta_spent = 0.0
        self.records = []

    def spend_sequential(self, eps, delta, note=""):
        self.eps_spent += eps
        self.delta_spent += delta
        self.records.append((eps, delta, note))

    def spend_sequential_charged_once(self, eps, delta, n_branches, note=""):
        """
        NOT true parallel composition.
        This method exists to record a sequential charge where the caller has
        already applied parallel composition across disjoint branches and is
        passing the effective per-block (eps, delta) once. It calls
        spend_sequential to keep the ledger strict and conservative.
        """
        self.spend_sequential(eps, delta, note=f"parallel({n_branches}) charged once: {note}")

    @property
    def total_eps(self):
        return self.eps_spent

    @property
    def total_delta(self):
        return self.delta_spent


def scale_with_public_bounds(X, lo, hi):
    """Clip to DP-released bounds and scale to [0,1]. Public, no extra budget."""
    X_arr = X.values if hasattr(X, "values") else np.asarray(X)
    span = np.maximum(hi - lo, 1e-6)
    cols = X.columns if hasattr(X, "columns") else None
    idx = X.index if hasattr(X, "index") else None
    return pd.DataFrame(np.clip((X_arr - lo) / span, 0, 1), columns=cols, index=idx)


def dp_sample_synthetic_records(balls, sizes, rng, d, sampling_mode="uniform"):
    """Generate synthetic records from each ball's released box geometry.

    sampling_mode:
      "uniform"  (default, ORIGINAL mechanism, unchanged) — sample uniformly
                 within the box. Costs zero extra privacy budget.
      "gaussian" (NEW, ablation) — sample from a diagonal Gaussian centered
                 at the box midpoint, std = box_width / 6, then clipped back
                 into the box. Mean/std are a DETERMINISTIC function of the
                 already-released box_lo/box_hi (public post-processing of
                 already-released quantities) — NOT a new noisy release of
                 the true per-ball mean/covariance. Costs exactly the same
                 privacy budget as "uniform": zero extra.
    """
    assert sampling_mode in ("uniform", "gaussian"), sampling_mode
    X_synth, y_synth = [], []

    for ball, n_out in zip(balls, sizes):
        if n_out <= 0:
            continue

        box_lo = ball.get("box_lo", np.zeros(d))
        box_hi = ball.get("box_hi", np.ones(d))
        box_hi = np.maximum(box_hi, box_lo)  # guarantee high >= low

        if sampling_mode == "uniform":
            X_new = rng.uniform(low=box_lo, high=box_hi, size=(int(n_out), d))
        else:  # "gaussian"
            center = (box_lo + box_hi) / 2.0
            width = np.maximum(box_hi - box_lo, 1e-6)
            std = width / 6.0  # ~99.7% mass inside the box pre-clipping
            X_new = rng.normal(loc=center, scale=std, size=(int(n_out), d))
            X_new = np.clip(X_new, box_lo, box_hi)

        X_synth.append(X_new)

        # ---- FIX D: sample labels using DP-released classes + noisy counts ----
        classes = ball.get("classes")
        noisy_counts = ball.get("noisy_class_counts")
        if classes is not None and noisy_counts is not None:
            counts = np.clip(np.asarray(noisy_counts, dtype=float), 0, None)
            total = counts.sum()
            if total > 0:
                probs = counts / total
                y_new = rng.choice(classes, size=int(n_out), p=probs)   # actual labels
            else:
                y_new = np.full(int(n_out), classes[0])
        else:
            y_new = np.zeros(int(n_out), dtype=int)
        # ----------------------------------------------------------------

        y_synth.extend(y_new)

    # NOTE (Fix 6): The forced-class injection safeguard has been removed.
    # Instead, class collapse will be reported as a diagnostic metric.

    if len(X_synth) == 0:
        return np.empty((0, d)), np.empty(0, dtype=int)

    X_out = np.vstack(X_synth)
    y_out = np.asarray(y_synth, dtype=int)
    return X_out, y_out


def run_dp_gbc_pipeline(
    X, y, seed, lo, hi, max_depth=4,
    eps_count_per_level=0.5,
    eps_split_per_level=0.5,
    eps_center_per_level=0.1,
    eps_radius_per_level=0.1,
    eps_count_release=0.5,
    delta=1e-5,
    all_classes=None,
    precomputed_split=None,
    sampling_mode="uniform",
    purity_tau=0.90,                       # <-- NEW
):
    rng = np.random.default_rng(seed)
    ledger = PrivacyLedger()

    # Train/test split: use precomputed_split if provided, else split internally
    if precomputed_split is not None:
        X_tr, X_te, y_tr, y_te = precomputed_split
    else:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y
        )

    # FIX B: public-bounds scaling (no private MinMax)
    X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
    X_te_n = scale_with_public_bounds(X_te, lo, hi)

    # DP granular balls, now with radius + class counts
    balls = dp_generate_granular_balls_with_radius(
        X_tr_n, y_tr.values,
        max_depth=max_depth,
        eps_count_per_level=eps_count_per_level,
        eps_split_per_level=eps_split_per_level,
        eps_radius_per_level=eps_radius_per_level,
        eps_center_per_level=eps_center_per_level,
        rng=rng,
        all_classes=all_classes,
        purity_tau=purity_tau,                  # <-- NEW
)

    # Granulation budget: count+split for D depths, count-only at terminal depth
    per_level_full = eps_count_per_level + eps_split_per_level
    for depth in range(max_depth):
        ledger.spend_sequential_charged_once(
            per_level_full, delta=0.0, n_branches=len(balls),
            note=f"granulation depth {depth+1} (count+split)"
        )
    ledger.spend_sequential_charged_once(
        eps_count_per_level, delta=0.0, n_branches=len(balls),
        note="granulation terminal depth (count only)"
    )

    # DP-release how many synthetic rows each ball should emit.
    # Sensitivity = 1 (one record changes one ball size by at most 1).
    ledger.spend_sequential(                              
        eps_count_release, 0.0,
        note="per-ball synthetic-size release (pure-eps)"
    )
    sizes = dp_release_ball_counts(balls, eps_count_release, rng)
    # Generate synthetic training data from released ball statistics.
    # This is post-processing — zero additional privacy cost.
    X_synth, y_synth = dp_sample_synthetic_records(
        balls, sizes, rng, d=X_tr_n.shape[1],
        sampling_mode=sampling_mode          # <-- pass through
    )

    # Test set stays as real holdout, exactly like the oracle/flat baselines.
    X_tr_out = pd.DataFrame(X_synth, columns=X_tr_n.columns)
    y_tr_out = pd.Series(y_synth)

    # Compatibility: return a boundary rate.
    boundary_rate = (
        float(np.mean([b["is_boundary"] for b in balls]))
        if balls else 0.0
    )

    return X_tr_out, X_te_n, y_tr_out, y_te, ledger, boundary_rate, balls

In [ ]:
# Cell 4c (UPDATED — Fix 1)
# NOTE: This is exactly run_dp_gbc_pipeline. Reported as "DP-GBC +LR head" in the paper.
# DP-LocalClip is now an alias for DP-GBC. No separate mechanism.
# Both pipelines produce identical DP synthetic releases.
# This cell only defines the alias; PrivacyLedger is assumed to be defined elsewhere
# (e.g., Cell 1 or wherever it was originally defined).

def run_dp_localclip_pipeline(
    X, y, seed, lo, hi, max_depth=3,
    eps_count_per_level=0.1,
    eps_split_per_level=0.1,
    eps_radius_per_level=0.1,
    eps_center_per_level=0.1,
    eps_noise=1.5,
    delta=1e-5,
    all_classes=None,
    precomputed_split=None,
    sampling_mode="uniform",          # <-- NEW
):
    """
    ALIAS, not a separate mechanism: verified line-for-line identical to
    run_dp_gbc_pipeline (same granular-ball generation, same per-ball count
    release, same box-uniform synthetic sampling). Kept only so existing call
    sites (Cells 8, 14, 16, 23, 25) don't need editing. It forwards to
    run_dp_gbc_pipeline so "DP-GBC" and "DP-LocalClip" numbers can never
    silently diverge into two different mechanisms again.
    In the paper: ONE mechanism ("DP-GBC synthetic release"), evaluated with
    different downstream heads (direct-ball / +LR / +MLP) -- never framed as
    two competing privacy mechanisms.
    """
    X_tr_out, X_te_n, y_tr_out, y_te, ledger, boundary_rate, balls = run_dp_gbc_pipeline(
        X, y, seed, lo, hi,
        max_depth=max_depth,
        eps_count_per_level=eps_count_per_level,
        eps_split_per_level=eps_split_per_level,
        eps_center_per_level=eps_center_per_level,
        eps_radius_per_level=eps_radius_per_level,
        eps_count_release=eps_noise,
        delta=delta,
        all_classes=all_classes,
        precomputed_split=precomputed_split,
        sampling_mode=sampling_mode,          # <-- pass through
    )
    mean_radius = float(np.mean([b["radius"] for b in balls])) if balls else float("nan")
    return X_tr_out, X_te_n, y_tr_out, y_te, ledger, mean_radius

In [ ]:
# =============================================================================
# Cell 4d: Exact budget allocator (SINGLE SOURCE OF TRUTH)
#
# The DP-GBC and KD-tree builders both perform, for a tree of max_depth D:
#   - (D + 1) count releases at eps_count each  (depths 0..D; terminal still counts)
#   - D       split releases at eps_split each  (depths 0..D-1)
#   - 1       per-ball synthetic-size release at eps_count_release
# so the honest total is:
#   total_eps = (D+1)*eps_count + D*eps_split + eps_count_release
#
# Every cell that previously computed
#     eps_count_release = TOTAL_EPS - D*(eps_count+eps_split)
# must call allocate_budget_exact() instead.
# =============================================================================

def allocate_budget_exact(total_eps, max_depth, eps_count=0.1, eps_split=0.1,
                          min_release=0.02):
    """
    Given a fixed per-level eps_count / eps_split (design choice preserved),
    return the residual eps_count_release so that the total matches exactly.

    Raises ValueError if total_eps is too small to fund the granulation
    phase at the requested per-level epsilons.
    """
    granulation = (max_depth + 1) * eps_count + max_depth * eps_split
    eps_count_release = total_eps - granulation
    if eps_count_release < min_release:
        raise ValueError(
            f"total_eps={total_eps} too small: granulation costs "
            f"{granulation:.4f}, leaving only {eps_count_release:.4f} "
            f"for per-ball release (min {min_release}). "
            f"Reduce max_depth or per-level epsilons."
        )
    return eps_count, eps_split, eps_count_release


# --- sanity check at the main operating point ---
_c, _s, _rel = allocate_budget_exact(3.0, max_depth=3)
_implied = 4 * _c + 3 * _s + _rel
assert abs(_implied - 3.0) < 1e-12, (_implied,)
print(f"[allocator] at eps=3.0, D=3: count={_c}  split={_s}  release={_rel:.4f}  "
      f"-> exact total={_implied:.6f}")

In [ ]:
# Cell 4e: advanced composition bound (Dwork, Rothblum & Vadhan 2010, Thm 3.20).
# Reporting-only helper — not called by any mechanism in this notebook.
# For k mechanisms each (eps,0)-DP composed adaptively:
#     eps_total(delta) = sqrt(2k ln(1/delta))*eps + k*eps*(e^eps - 1)
# The naive linear bound is k*eps. At our operating point (eps=0.1, k=7) the
# advanced bound is LOOSER than naive summing — this is expected for small
# eps and small k; we report it only to document that we checked.
def advanced_composition_eps(eps_per_mechanism, k, delta_target):
    if k <= 1:
        return eps_per_mechanism, 0.0
    term = np.sqrt(2 * k * np.log(1.0 / delta_target)) * eps_per_mechanism
    term += k * eps_per_mechanism * (np.exp(eps_per_mechanism) - 1)
    return float(term), float(delta_target)

# DP-GBC granulation phase = (D+1)=4 count + D=3 split = 7 pure-eps mechanisms.
eps_naive = 7 * 0.1
eps_adv, delta_adv = advanced_composition_eps(0.1, k=7, delta_target=1e-6)
print(f"DP-GBC granulation phase, naive linear composition: eps={eps_naive:.3f}, delta=0")
print(f"DP-GBC granulation phase, advanced composition:     eps={eps_adv:.3f}, delta={delta_adv:.0e}")

In [ ]:
# Cell 5 (UPDATED): Imports only.
#
# The evaluate_utility() function that used to live here was fully shadowed by
# Cell 7's redefinition (Python rebinds the name, so every cell after Cell 7
# used Cell 7's version). That was confusing: this cell documented
# class-balanced oversampling for the MLP, but Cell 7's version silently
# dropped it, so the LR-vs-MLP comparison inside Cell 7 was not actually
# apples-to-apples. Cell 7 has since been fixed to restore the oversampling,
# and the single authoritative evaluate_utility() now lives there.
#
# The two imports below are kept because they are reused in Cell 7 (and
# elsewhere), so removing this cell entirely would break those cells.

from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample

# Cell 6: Attacks, PLUS a sanity check that the attack actually works when
# there IS something to detect. Without this control, an MIA_AUC near 0.5
# under DP is uninterpretable -- you can't tell "well protected" apart from
# "the attack doesn't work."

def membership_inference_attack(model, X_tr, y_tr, X_te, y_te):
    conf_tr = np.max(model.predict_proba(X_tr), axis=1)
    conf_te = np.max(model.predict_proba(X_te), axis=1)
    scores = np.concatenate([conf_tr, conf_te])
    labels = np.concatenate([np.ones(len(conf_tr)), np.zeros(len(conf_te))])
    try:
        return roc_auc_score(labels, scores)
    except Exception:
        return 0.5

def positive_control_mia(X_tr, y_tr, X_te, y_te, seed):
    """Deliberately overfit an unregularized high-capacity RF on RAW
    (non-private) data. If MIA_AUC here isn't well above 0.5, your attack
    implementation itself is the problem, not the privacy mechanism."""
    overfit_model = RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=1, random_state=seed
    )
    overfit_model.fit(X_tr, y_tr)
    return membership_inference_attack(overfit_model, X_tr, y_tr, X_te, y_te)

def membership_inference_attack_rf(X_tr, y_tr, X_te, y_te, seed):
    """Confidence-gap membership inference attack (Yeom et al. 2018 style):
    trains a classifier directly on the target's own training set and
    measures whether its confidence differs between members (X_tr) and
    non-members (X_te). NOT a Shokri et al. (2017) shadow-model attack --
    no separate model trained on disjoint data is used here."""
    # Confidence-gap model: RF trained on train set
    model = RandomForestClassifier(
        n_estimators=100, max_depth=6,
        class_weight="balanced", random_state=seed
    )
    model.fit(X_tr, y_tr)
    # Features: confidence scores from the model
    conf_tr = np.max(model.predict_proba(X_tr), axis=1)
    conf_te = np.max(model.predict_proba(X_te), axis=1)
    scores = np.concatenate([conf_tr, conf_te])
    labels = np.concatenate([np.ones(len(conf_tr)), np.zeros(len(conf_te))])
    try:
        auc = roc_auc_score(labels, scores)
    except Exception:
        auc = 0.5
    return auc, model

In [ ]:
# Cell 6b (NEW): Attribute Inference Attack, restored from your original design
# but fixed: uses a fixed seed correctly, class_weight balanced, and reports
# lift over the trivial majority-class baseline rather than raw accuracy alone.

from sklearn.preprocessing import KBinsDiscretizer

def attribute_inference_attack(X_tr_attack, X_te_attack, sens_tr, sens_te, seed):
    sens_tr = np.nan_to_num(np.asarray(sens_tr).ravel(), nan=0.0)
    sens_te = np.nan_to_num(np.asarray(sens_te).ravel(), nan=0.0)

    if len(np.unique(sens_tr)) > 10:  # continuous -> discretize into 3 bins
        disc = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
        sens_tr = disc.fit_transform(sens_tr.reshape(-1, 1)).ravel().astype(int)
        sens_te = disc.transform(sens_te.reshape(-1, 1)).ravel().astype(int)
    else:
        sens_tr, sens_te = sens_tr.astype(int), sens_te.astype(int)

    if len(np.unique(sens_tr)) < 2:
        print(f"    [AIA Note] Sensitive attribute collapsed to 1 bin due to noise; returning 0.0 lift by construction.")
        majority = np.bincount(sens_tr).argmax()
        preds = np.full_like(sens_te, majority)
        baseline = pd.Series(sens_te).value_counts(normalize=True).max()
        return accuracy_score(sens_te, preds) - baseline

    attacker = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=seed)
    attacker.fit(X_tr_attack, sens_tr)
    preds = attacker.predict(X_te_attack)
    baseline = pd.Series(sens_te).value_counts(normalize=True).max()
    return accuracy_score(sens_te, preds) - baseline  # "lift" over guessing majority sensitive value

def run_aia(X_tr_noisy, y_tr, X_te_noisy, y_te, model, sens_col_idx=0, seed=SEED):
    """Attack tries to infer feature[sens_col_idx] from the OTHER features + model outputs."""
    non_sens_cols = [c for i, c in enumerate(X_tr_noisy.columns) if i != sens_col_idx]
    X_tr_attack = np.hstack([X_tr_noisy[non_sens_cols].values, model.predict_proba(X_tr_noisy)])
    X_te_attack = np.hstack([X_te_noisy[non_sens_cols].values, model.predict_proba(X_te_noisy)])
    sens_tr = X_tr_noisy.iloc[:, sens_col_idx].values
    sens_te = X_te_noisy.iloc[:, sens_col_idx].values
    return attribute_inference_attack(X_tr_attack, X_te_attack, sens_tr, sens_te, seed)

In [ ]:
# Cell 6c (NEW): Correct MIA protocol for SYNTHETIC-data conditions.
# Per van Breugel et al. (AISTATS 2023) and TAPAS (Houssiau et al. 2022):
# membership is a property of REAL records. The attacker may only ever query
# real candidate points -- never synthetic rows, which have no 1:1 mapping
# to any actual individual and are not a valid member/non-member proxy.
def membership_inference_attack_rf_synth(X_tr_synth, y_tr_synth,
                                          X_real_members, y_real_members,
                                          X_real_nonmembers, y_real_nonmembers,
                                          seed):
    shadow = RandomForestClassifier(
        n_estimators=100, max_depth=6,
        class_weight="balanced", random_state=seed
    )
    shadow.fit(X_tr_synth, y_tr_synth)
    conf_members = np.max(shadow.predict_proba(X_real_members), axis=1)
    conf_nonmembers = np.max(shadow.predict_proba(X_real_nonmembers), axis=1)
    scores = np.concatenate([conf_members, conf_nonmembers])
    labels = np.concatenate([np.ones(len(conf_members)), np.zeros(len(conf_nonmembers))])
    try:
        return roc_auc_score(labels, scores), shadow
    except Exception:
        return 0.5, shadow

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, average_precision_score

TOTAL_EPS = 3.0
SEEDS = [42, 123, 456, 789, 101]

NUMERIC_DATASETS = ["Banknote","BreastCancer","Ionosphere","MAGIC04",
                    "PimaDiabetes","Sonar","Wine"]


def evaluate_utility(X_tr, y_tr, X_te, y_te):
    """Logistic regression; returns balanced accuracy (and AUPRC for binary)."""
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    out = {
        "balanced_accuracy": balanced_accuracy_score(y_te, preds),
        "auprc": np.nan,
    }
    try:
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X_te)
            if probs is not None and len(np.unique(y_te)) == 2 and probs.shape[1] == 2:
                out["auprc"] = average_precision_score(y_te, probs[:, 1])
    except Exception as e:
        print("AUPRC skipped:", e)
    return out, model


print(f"Cell 7-SLIM loaded. TOTAL_EPS={TOTAL_EPS}, "
      f"SEEDS={SEEDS}, NUMERIC_DATASETS={len(NUMERIC_DATASETS)} datasets.")
print(f"Budget check: {allocate_budget_exact(3.0, 3, 0.1, 0.1)}")

output
Cell 7-SLIM loaded. TOTAL_EPS=3.0, SEEDS=[42, 123, 456, 789, 101], NUMERIC_DATASETS=7 datasets.
Budget check: (0.1, 0.1, 2.3)

# ===== Ball tie-in ablation: split eps_release between size and mean =====
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

PRIMARY = ["Banknote","BreastCancer","Ionosphere","MAGIC04",
                    "PimaDiabetes","Sonar","Wine"]

tie_rows = []
for eps_mean_frac in [0.0, 0.25, 0.5]:      # 0.0 = uniform baseline (current)
    eps_size     = 2.3 * (1 - eps_mean_frac)
    eps_mean     = 2.3 * eps_mean_frac
    for ds_name in PRIMARY:
        if ds_name not in datasets:
            continue
        X, y = datasets[ds_name]
        lo, hi = public_bounds_for(ds_name, X)
        for seed in SEEDS:
            try:
                X_tr, X_te, y_tr, y_te = train_test_split(
                    X, y, test_size=0.3, random_state=seed, stratify=y)
            except ValueError:
                continue
            X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
            X_te_n = scale_with_public_bounds(X_te, lo, hi)
            rng = np.random.default_rng(seed)
            balls = dp_generate_granular_balls_with_radius(
                X_tr_n, y_tr.values, max_depth=3,
                eps_count_per_level=0.1, eps_split_per_level=0.1,
                eps_radius_per_level=0.0, eps_center_per_level=0.0,
                min_size=10, rng=rng, all_classes=classes_public[ds_name])
            sizes = dp_release_ball_counts(balls, eps_size, rng)

            if eps_mean_frac == 0.0:
                X_s, y_s = dp_sample_synthetic_records(
                    balls, sizes, rng, d=X_tr_n.shape[1], sampling_mode="uniform")
            else:
                means = dp_release_box_means(balls, eps_mean, rng, X_tr_n.values)
                X_s, y_s = dp_sample_truncated_gaussian(
                    balls, sizes, means, rng, d=X_tr_n.shape[1])

            if len(X_s) == 0 or pd.Series(y_s).nunique() < 2:
                continue
            util, _ = evaluate_utility(
                pd.DataFrame(X_s, columns=X_tr_n.columns), pd.Series(y_s),
                X_te_n, y_te)
            tie_rows.append({"Dataset": ds_name, "Seed": seed,
                             "eps_mean_frac": eps_mean_frac,
                             "bal_acc": util["balanced_accuracy"]})

df_tie = pd.DataFrame(tie_rows)
df_tie.to_csv("dpgbc_ball_tiein.csv", index=False)
print(df_tie.groupby("eps_mean_frac")["bal_acc"].agg(["mean","std","count"]).round(3).to_string())

In [ ]:
# Cell 19 (FIXED): DP-PCA baseline (input-perturbation family: BDMS05 / AGTZ14)
# Bug 1: row clipping now shrinks over-norm rows (norm ≤ 1) instead of normalizing all rows.
# Bug 2: Laplace noise on projected Z now correctly uses L1 sensitivity sqrt(k).
# FIX (public bounds): private MinMaxScaler removed; use public_bounds_for.

def dp_pca_transform(X_tr, X_te, eps, n_components, rng):
    Xc = X_tr.values - 0.5   # fixed PUBLIC center (data is [0,1] → midpoint 0.5)
    norms = np.linalg.norm(Xc, axis=1, keepdims=True)
    # Bug 1 fix: clip rows whose L2 norm > 1 to unit norm; leave others untouched
    clip_scale = np.minimum(1.0, 1.0 / np.maximum(norms, 1e-8))
    Xc_unit = Xc * clip_scale
    A = Xc_unit.T @ Xc_unit
    d = A.shape[0]
    noise = rng.normal(0, 2.0 * np.sqrt(2*np.log(1.25/1e-5)) / max(eps, 1e-6), size=(d, d))
    noise = np.triu(noise); noise = noise + noise.T - np.diag(np.diag(noise))
    A_noisy = (A + noise); A_noisy = (A_noisy + A_noisy.T) / 2
    eigvals, eigvecs = np.linalg.eigh(A_noisy)
    proj = eigvecs[:, np.argsort(eigvals)[::-1][:n_components]]
    Xte_c = X_te.values - 0.5
    return Xc_unit @ proj, Xte_c @ proj

dppca_results = []
for ds_name, (X, y) in datasets.items():
    # Public bounds for this dataset (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        except ValueError:
            continue

        # Public-bounds scaling (replaces private MinMaxScaler)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        eps_pca = TOTAL_EPS * 0.4
        eps_noise = TOTAL_EPS - eps_pca
        k = max(1, min(5, X.shape[1] - 1))
        Z_tr, Z_te = dp_pca_transform(X_tr_n, X_te_n, eps_pca, k, rng)

        # L2 sensitivity = 1 — code is already correct; the previous comment was wrong.
        delta_z = 1e-5
        sigma_z = 1.0 * np.sqrt(2 * np.log(1.25 / delta_z)) / max(eps_noise, 1e-6)  # L2 sensitivity = 1
        Z_tr = np.clip(Z_tr + rng.normal(0, sigma_z, Z_tr.shape), -1, 1)
        Z_te = np.clip(Z_te, -1, 1)   # test set isn't part of the private release -- don't noise it

        Z_tr_df = pd.DataFrame(Z_tr, columns=[f"pc{i}" for i in range(Z_tr.shape[1])])
        Z_te_df = pd.DataFrame(Z_te, columns=[f"pc{i}" for i in range(Z_te.shape[1])])
        util, _ = evaluate_utility(Z_tr_df, y_tr.reset_index(drop=True), Z_te_df, y_te.reset_index(drop=True))
        mia, _ = membership_inference_attack_rf(Z_tr_df, y_tr.reset_index(drop=True), Z_te_df, y_te.reset_index(drop=True), seed)
        dppca_results.append({"Dataset": ds_name, "Seed": seed,
                               "dppca_bal_acc": util["balanced_accuracy"], "dppca_mia_rf": mia})

print(pd.DataFrame(dppca_results).groupby("Dataset")[["dppca_bal_acc","dppca_mia_rf"]].mean().round(3).to_string())

In [ ]:
# Cell 20 (NEW): DP-SGD / gradient-perturbation baseline (Abadi et al. 2016;
# Chaudhuri, Monteleoni, Sarwate 2011). We implement FULL-BATCH DP-GD, not the
# Poisson-subsampled minibatch version -- subsampling amplification needs a
# moments accountant / RDP subsampling theorem that's easy to misstate from
# memory. Full-batch DP-GD has clean, verifiable zCDP accounting: each
# iteration is a Gaussian release of a clipped-and-averaged gradient with
# sensitivity C/n, so rho_per_iter = 1/(2*sigma^2) zCDP (sigma = noise
# multiplier), additive over T iterations, converted via Bun & Steinke 2016's
# zCDP->DP formula. Binary classification only (noted below).

def zcdp_to_eps(rho, delta):
    return rho + 2 * np.sqrt(rho * np.log(1 / delta))

def dp_class_weights(y_tr, eps_count, rng):
    classes = np.unique(y_tr)
    n = len(y_tr)
    noisy_counts = {}
    for c in classes:
        true_count = np.sum(y_tr == c)
        noisy_counts[c] = max(1.0, true_count + rng.laplace(0, 1.0 / max(eps_count, 1e-6)))
    weights = {c: n / (len(classes) * cnt) for c, cnt in noisy_counts.items()}
    return weights, noisy_counts

def dp_sgd_logreg(X_tr, y_tr, eps_target, delta, T=100, clip_norm=1.0, lr=0.5, rng=None,
                   max_class_weight=3.0, eps_count_for_weights=0.05):
    rng = rng or np.random.default_rng(0)
    n = X_tr.shape[0]
    Xb = np.hstack([X_tr, np.ones((n, 1))])
    w = np.zeros(Xb.shape[1])
    y_pm = np.where(y_tr == y_tr.min(), -1, 1)

    # class weights now DP-released, spending a small slice of the budget
    eps_target_gd = eps_target - eps_count_for_weights  # remaining budget for the actual training
    raw_weight, noisy_counts = dp_class_weights(y_tr, eps_count_for_weights, rng)
    mean_w = sum(noisy_counts[c] * raw_weight[c] for c in noisy_counts) / sum(noisy_counts.values())
    sample_w = np.clip(np.array([raw_weight[v] for v in y_tr]) / mean_w, 0.0, max_class_weight)
    effective_clip = clip_norm * max_class_weight

    lo, hi = 0.1, 100.0
    for _ in range(40):
        mid = (lo + hi) / 2
        rho = T / (2 * mid**2)
        if zcdp_to_eps(rho, delta) > eps_target_gd: lo = mid
        else: hi = mid
    sigma = hi

    for t in range(T):
        p = 1 / (1 + np.exp(-y_pm * (Xb @ w)))
        grad_i = (-(1 - p) * y_pm)[:, None] * Xb                     # raw gradient, UNWEIGHTED
        norms = np.linalg.norm(grad_i, axis=1, keepdims=True)
        grad_i = grad_i * np.minimum(1.0, clip_norm / np.maximum(norms, 1e-8))  # clip FIRST
        grad_avg = (sample_w[:, None] * grad_i).mean(axis=0)          # weight applied in the AVERAGE
        w = w - lr * (grad_avg + rng.normal(0, effective_clip * sigma / n, size=grad_avg.shape))
    return w, sigma

def dp_sgd_predict_proba(w, X):
    Xb = np.hstack([X, np.ones((X.shape[0], 1))])
    p1 = 1 / (1 + np.exp(-(Xb @ w)))
    return np.stack([1 - p1, p1], axis=1)

dpsgd_results = []
for ds_name, (X, y) in datasets.items():
    if y.nunique() != 2:
        continue

    # Public bounds for this dataset (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        except ValueError:
            continue

        # Public-bounds scaling (replaces private MinMaxScaler)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        # CHANGE: T from 100 -> 25 (5th positional argument)
        w, sigma_used = dp_sgd_logreg(X_tr_n.values, y_tr.values, TOTAL_EPS, 1e-5, 100, 1.0, 0.5, rng)
        preds = dp_sgd_predict_proba(w, X_te_n.values).argmax(axis=1) + y_tr.min()
        bal_acc = balanced_accuracy_score(y_te, preds)
        if bal_acc == 0.5:
            print(f"{ds_name}: w norm = {np.linalg.norm(w):.4f}, "
                  f"noise std per step = {1.0 * sigma_used / len(X_tr_n):.4f}")

        conf_tr = np.max(dp_sgd_predict_proba(w, X_tr_n.values), axis=1)
        conf_te = np.max(dp_sgd_predict_proba(w, X_te_n.values), axis=1)
        try:
            mia = roc_auc_score(np.concatenate([np.ones(len(conf_tr)), np.zeros(len(conf_te))]),
                                 np.concatenate([conf_tr, conf_te]))
        except Exception:
            mia = 0.5
        dpsgd_results.append({"Dataset": ds_name, "Seed": seed, "dpsgd_bal_acc": bal_acc,
                               "dpsgd_mia": mia, "sigma_used": sigma_used})

n_multiclass = sum(1 for _, (_, y) in datasets.items() if y.nunique() != 2)
# ADD: include sigma_used in the printed summary and assign dataframe to variable
df_dpsgd_out = pd.DataFrame(dpsgd_results)
print(df_dpsgd_out.groupby("Dataset")[["dpsgd_bal_acc","dpsgd_mia","sigma_used"]].mean().round(3).to_string())
print(f"\nbinary-only baseline -- {n_multiclass} multiclass datasets excluded, state this in limitations")
print("DP-SGD-LR baseline: T=100 resolves the earlier BankMarketing iteration-limit "
      "(0.500 -> 0.627). CreditFraud remains at chance even at T=100 with near-zero "
      "noise (isolation test), consistent with its extreme 500:1 class imbalance "
      "exceeding what max_class_weight=3.0 can compensate for -- a separate, unresolved "
      "limitation of this baseline, not an iteration-count artifact.")

In [ ]:
# Cell 22 (FIXED): Reconstruction attack.
# Uses public schema bounds for scaling, not private MinMaxScaler, to avoid
# leaking bounds twice and inflating the attack_improvement metric.

from sklearn.neural_network import MLPRegressor

recon_results = []
for ds_name, (X, y) in datasets.items():
    # Public bounds for this dataset (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=seed, stratify=y)
        except ValueError:
            continue
        X_priv, X_aux = train_test_split(X_tr, test_size=0.5, random_state=seed)

        # Public-bounds scaling (replaces MinMaxScaler().fit(X_tr))
        X_priv_n = scale_with_public_bounds(X_priv, lo, hi)
        X_aux_n = scale_with_public_bounds(X_aux, lo, hi)

        rng2 = np.random.default_rng(seed + 1)
        X_priv_flat = np.clip(X_priv_n + rng2.laplace(0, 1.0 / TOTAL_EPS, X_priv_n.shape), 0, 1)
        X_aux_flat = np.clip(X_aux_n + rng2.laplace(0, 1.0 / TOTAL_EPS, X_aux_n.shape), 0, 1)

        denoiser = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=500, random_state=0)
        denoiser.fit(X_aux_flat, X_aux_n)
        X_recon = denoiser.predict(X_priv_flat)

        mse_recon = np.mean((X_recon - X_priv_n) ** 2)
        mse_naive = np.mean((X_priv_flat - X_priv_n) ** 2)
        recon_results.append({"Dataset": ds_name, "Seed": seed, "mse_reconstructed": mse_recon,
                               "mse_naive_noised": mse_naive, "attack_improvement": mse_naive - mse_recon})

df_recon = pd.DataFrame(recon_results)
print(df_recon.groupby("Dataset")[["mse_naive_noised","mse_reconstructed","attack_improvement"]].mean().round(4).to_string())
print("\npositive attack_improvement = denoiser meaningfully beat raw noised values (privacy concern)")

In [ ]:
# Cell 23 (UPDATED): Offline population-calibrated MIA (Carlini et al., IEEE S&P 2022, LiRA).
#
# Fixes:
#   - public schema bounds instead of private dp_release_feature_bounds
#   - updated run_dp_localclip_pipeline signature with lo, hi
#   - explicit eps_center_per_level
#   - budget still sums to TOTAL_EPS (via allocate_budget_exact, matching Cells 7/8/etc.)
#   - NEW: full 5-seed sweep (was SEEDS[0] only).
#
# COMPUTE-TIME WARNING (see Part E):
#   LiRA trains n_shadow=16 shadow models PER (dataset, seed). With 14 datasets
#   x 5 seeds that is 14*5*16 = 1120 shadow LR fits, PLUS 14*5 = 70 DP-LocalClip
#   pipeline runs. On Kaggle this can take many hours and risks a mid-run
#   disconnect. If you only need a rough number, set SEEDS_LIRA = SEEDS[:1] or
#   SEEDS[:2] below. The loop is written so you can swap in any subset.
#
# NOTE (CAVEAT): The shadow models in offline_lira are trained with flat Laplace noise,
# but the target model here is trained on the DP-LocalClip synthetic release.
# This violates LiRA's assumption that shadow and target models use the same mechanism.
# This exploratory cell is retained as-is for a rough estimate only; do not use
# these results as a rigorous privacy claim without rewriting offline_lira to
# train shadow models via run_dp_localclip_pipeline per shadow trial.

def offline_lira(X_pool, y_pool, target_idx, n_shadow, eps, rng):
    n = len(X_pool)
    out_conf = {i: [] for i in target_idx}
    for s in range(n_shadow):
        rs = np.random.default_rng(rng.integers(1e9))
        keep = rs.choice(n, size=n // 2, replace=False)
        keep_set = set(keep.tolist())
        X_shadow, y_shadow = X_pool[keep], y_pool[keep]
        X_shadow_flat = np.clip(X_shadow + rs.laplace(0, 1.0 / eps, X_shadow.shape), 0, 1)
        if len(np.unique(y_shadow)) < 2:
            continue
        model = LogisticRegression(max_iter=500, class_weight="balanced", random_state=s)
        model.fit(X_shadow_flat, y_shadow)
        for i in target_idx:
            if i not in keep_set:
                x_n = np.clip(
                    X_pool[i:i + 1] + rs.laplace(0, 1.0 / eps, X_pool[i:i + 1].shape),
                    0, 1
                )
                conf = model.predict_proba(x_n)[0, y_pool[i]]
                out_conf[i].append(np.log(conf / max(1 - conf, 1e-8)))
    return out_conf


TOTAL_EPS = float(globals().get("TOTAL_EPS", 3.0))

# ---- Compute-time control: edit to SEEDS[:1] or SEEDS[:2] for a fast run ----
SEEDS_LIRA = SEEDS            # <-- full 5-seed sweep; change to SEEDS[:2] if time-limited

NUMERIC_DATASETS = ["Banknote","BreastCancer","Ionosphere","MAGIC04",
                    "PimaDiabetes","Sonar","Wine"]

lira_results = []

for ds_name, (X, y) in datasets.items():
    if ds_name not in NUMERIC_DATASETS:
        continue

    lo, hi = public_bounds_for(ds_name, X)   # no privacy budget spent

    for seed in SEEDS_LIRA:                  # <-- full 5-seed sweep (was SEEDS[0])
        rng = np.random.default_rng(seed)

        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue

        # Public-bounds scaling — no private MinMax
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi).values
        X_te_n = scale_with_public_bounds(X_te, lo, hi)
        y_tr_arr = y_tr.values

        target_idx = rng.choice(
            len(X_tr_n), size=min(40, len(X_tr_n)), replace=False
        )

        # Offline LiRA shadow-model phase
        out_conf = offline_lira(
            X_tr_n,
            y_tr_arr,
            target_idx,
            n_shadow=16,
            eps=TOTAL_EPS,
            rng=rng
        )

        # Fixed DP-LocalClip synthetic release
        eps_cnt = eps_spl = 0.1
        eps_ctr = 0.0   # public
        eps_rad = 0.0   # public
        _, _, eps_noise_d = allocate_budget_exact(TOTAL_EPS, 3, eps_cnt, eps_spl)

        X_tr_d, X_te_d, y_tr_d, y_te_d, _, _ = run_dp_localclip_pipeline(
            X, y, seed, lo, hi,
            max_depth=3,
            eps_count_per_level=eps_cnt,
            eps_split_per_level=eps_spl,
            eps_radius_per_level=eps_rad,
            eps_center_per_level=eps_ctr,
            eps_noise=eps_noise_d,
            all_classes=classes_public[ds_name]
        )

        if len(X_tr_d) == 0:
            continue

        _, target_model = evaluate_utility(X_tr_d, y_tr_d, X_te_d, y_te_d)

        correct, valid = 0, 0
        for i in target_idx:
            if len(out_conf[i]) < 3:
                continue
            mu = np.mean(out_conf[i])
            sd = np.std(out_conf[i]) + 1e-6

            # Original training point, public-bounds scaled, not noised
            obs_conf = target_model.predict_proba(X_tr_n[i:i + 1])[0, y_tr_arr[i]]
            obs_logit = np.log(obs_conf / max(1 - obs_conf, 1e-8))

            valid += 1
            if (obs_logit - mu) / sd > 0:
                correct += 1

        if valid:
            lira_results.append({
                "Dataset": ds_name,
                "Seed": seed,
                "n_targets": valid,
                "guess_member_rate": correct / valid,
            })
        print(f"done LiRA: {ds_name} / seed {seed}")

df_lira = pd.DataFrame(lira_results)
if len(df_lira):
    # Per-dataset mean across seeds
    print(
        df_lira.groupby("Dataset")[["n_targets", "guess_member_rate"]]
              .mean().round(3).to_string()
    )
    print(f"\nOverall mean guess_member_rate = {df_lira['guess_member_rate'].mean():.3f} "
          f"(0.5 = no distinguishability)")
else:
    print("No LiRA rows produced.")
print("\nguess_member_rate near 0.5 = attack can't distinguish member vs non-member")

In [ ]:
# Cell 24 (FIXED): Property inference attack
# Uses public schema bounds for scaling, not private MinMaxScaler.
# UPDATED: full 5-seed sweep per dataset (was a single SEED-global RNG stream).
# Results now carry "Seed"; summary averages over seeds per dataset.

from sklearn.linear_model import LinearRegression

def property_inference_attack(X, y, ds_name, eps, n_shadow, rng):
    X_df = X if hasattr(X, "values") else pd.DataFrame(X)
    y_arr = y.values if hasattr(y, "values") else np.asarray(y)
    # Public bounds from schema; no privacy budget spent
    lo, hi = public_bounds_for(ds_name, X_df)
    X_n = scale_with_public_bounds(X_df, lo, hi).values

    probe = X_n[rng.choice(len(X_n), size=min(50, len(X_n)), replace=False)]
    pos_val = y_arr.max()
    meta_X, meta_y = [], []
    for s in range(n_shadow):
        rs = np.random.default_rng(rng.integers(1e9))
        target_frac = rs.uniform(0.1, 0.9)
        pos_idx = np.where(y_arr == pos_val)[0]; neg_idx = np.where(y_arr != pos_val)[0]
        n_take = min(len(pos_idx), len(neg_idx), 200)
        n_pos = int(n_take * target_frac); n_neg = n_take - n_pos
        if n_pos < 2 or n_neg < 2: continue
        idx = np.concatenate([rs.choice(pos_idx, n_pos, replace=True), rs.choice(neg_idx, n_neg, replace=True)])
        X_shadow_flat = np.clip(X_n[idx] + rs.laplace(0, 1.0 / eps, (len(idx), X_n.shape[1])), 0, 1)
        model = LogisticRegression(max_iter=500, class_weight="balanced", random_state=s)
        model.fit(X_shadow_flat, y_arr[idx])
        probs = model.predict_proba(probe)
        meta_X.append([probs.mean(), probs.std(), np.percentile(probs[:, -1], 25), np.percentile(probs[:, -1], 75)])
        meta_y.append(target_frac)
    if len(meta_X) < 10: return None
    meta_X, meta_y = np.array(meta_X), np.array(meta_y)
    split = int(0.7 * len(meta_X))
    meta_model = LinearRegression().fit(meta_X[:split], meta_y[:split])
    mae = np.mean(np.abs(meta_model.predict(meta_X[split:]) - meta_y[split:]))
    baseline_mae = np.mean(np.abs(meta_y[split:] - meta_y[:split].mean()))
    return {"Dataset": ds_name, "attack_mae": mae, "baseline_mae": baseline_mae}

pia_results = []

for ds_name, (X, y) in datasets.items():
    if y.nunique() != 2:
        continue
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        res = property_inference_attack(X, y, ds_name, TOTAL_EPS, 60, rng)
        if res:
            res["Seed"] = seed
            pia_results.append(res)
        print(f"done PIA: {ds_name} / seed {seed}")

df_pia = pd.DataFrame(pia_results)
if len(df_pia):
    df_pia["attack_beats_baseline"] = df_pia["attack_mae"] < df_pia["baseline_mae"]

    # Per-dataset mean across seeds
    summary_pia = (
        df_pia.groupby("Dataset")[["attack_mae", "baseline_mae", "attack_beats_baseline"]]
              .mean().round(3)
    )
    print("\n=== Property-inference attack (per-dataset mean across seeds) ===")
    print(summary_pia.to_string())

    # Also show seed-level detail so you can eyeball variance
    print("\n=== Per-seed detail ===")
    print(df_pia[["Dataset", "Seed", "attack_mae", "baseline_mae", "attack_beats_baseline"]]
                .round(3).to_string(index=False))
else:
    print("No PIA rows produced.")

In [ ]:
# Cell 25 (REVISED): Bayesian AIA across flat, DP-GBC, and DP-LocalClip.
# Public bounds used instead of private DP-released bounds.

from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import KBinsDiscretizer

# Sensitive column mapping (10-dataset setup).
# Only datasets with a natural human-interpretable sensitive attribute listed.
# All other binary datasets fall through to column 0 via resolve_sens_col_idx.
SENSITIVE_COL = {
    "BreastCancer":   "mean radius",   # tumor size proxy
    "PimaDiabetes":   "age",
    "HeartCleveland": "age",           # renamed from HeartHungarian in Cell 2 rewrite
    "CreditG":        "age",
}

def resolve_sens_col_idx(X, ds_name):
    target = SENSITIVE_COL.get(ds_name)
    if target is None:
        return 0
    cols_lower = [c.lower() for c in X.columns]
    if target.lower() in cols_lower:
        return cols_lower.index(target.lower())
    print(f"WARNING: sensitive column '{target}' not found for {ds_name}; falling back to col 0")
    return 0


def bayesian_aia(X_tr_attack, X_te_attack, sens_tr, sens_te):
    """
    Same function as before. Returns accuracy - majority-class baseline.
    """
    sens_tr = np.nan_to_num(np.asarray(sens_tr).ravel())
    sens_te = np.nan_to_num(np.asarray(sens_te).ravel())

    if len(np.unique(sens_tr)) > 10:
        disc = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
        sens_tr = disc.fit_transform(sens_tr.reshape(-1, 1)).ravel().astype(int)
        sens_te = disc.transform(sens_te.reshape(-1, 1)).ravel().astype(int)
    else:
        sens_tr = sens_tr.astype(int)
        sens_te = sens_te.astype(int)

    if len(np.unique(sens_tr)) < 2:
        return 0.0

    clf = GaussianNB().fit(X_tr_attack, sens_tr)
    preds = clf.predict(X_te_attack)
    baseline = pd.Series(sens_te).value_counts(normalize=True).max()
    return accuracy_score(sens_te, preds) - baseline


TOTAL_EPS = float(globals().get("TOTAL_EPS", 3.0))

bayes_aia_results = []

for ds_name, (X, y) in datasets.items():
    if y.nunique() != 2:
        continue

    sens_col_idx = resolve_sens_col_idx(X, ds_name)
    sens_col_name = X.columns[sens_col_idx]

    # Public bounds from schema; no privacy budget spent
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue

        # Public-bounds scaling — not private MinMax
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        non_sens_cols = [c for c in X.columns if c != sens_col_name]

        # ------------------- Flat noise -------------------
        rng = np.random.default_rng(seed)
        X_tr_flat = make_flat_noised(X_tr_n, TOTAL_EPS, rng)
        X_te_flat = make_flat_noised(X_te_n, TOTAL_EPS, rng)

        flat_lift = bayesian_aia(
            X_tr_flat[non_sens_cols].values,
            X_te_flat[non_sens_cols].values,
            X_tr_n[sens_col_name].values,   # original scaled sensitive, not noised
            X_te_n[sens_col_name].values    # original scaled sensitive, not noised
        )

        # ------------------- DP-GBC -------------------
        max_depth = 3
        eps_count = eps_split = 0.1
        eps_center = 0.0   # public
        eps_radius = 0.0   # public
        _, _, eps_count_release = allocate_budget_exact(
            TOTAL_EPS, max_depth, eps_count, eps_split
        )

        X_tr_c, X_te_c, y_tr_c, y_te_c, ledger_c, _, balls_c = run_dp_gbc_pipeline(
            X, y, seed, lo, hi,          # FIX: was seed * 2
            max_depth=max_depth,
            eps_count_per_level=eps_count,
            eps_split_per_level=eps_split,
            eps_center_per_level=eps_center,
            eps_radius_per_level=eps_radius,
            eps_count_release=eps_count_release,
            all_classes=classes_public[ds_name]
        )

        dpgbc_lift = bayesian_aia(
            X_tr_c[non_sens_cols].values,
            X_te_c[non_sens_cols].values,
            X_tr_c[sens_col_name].values,   # synthetic sensitive, used as training target
            X_te_c[sens_col_name].values    # real scaled sensitive, used as test target
        )

        # ------------------- DP-LocalClip -------------------
        eps_noise_lc = eps_count_release  # same remaining budget after fixed redesign

        X_tr_lc, X_te_lc, y_tr_lc, y_te_lc, ledger_lc, _ = run_dp_localclip_pipeline(
            X, y, seed, lo, hi,          # FIX: was seed * 2 + 1
            max_depth=max_depth,
            eps_count_per_level=eps_count,
            eps_split_per_level=eps_split,
            eps_radius_per_level=eps_radius,
            eps_center_per_level=eps_center,
            eps_noise=eps_noise_lc,
            all_classes=classes_public[ds_name]
        )

        dplc_lift = bayesian_aia(
            X_tr_lc[non_sens_cols].values,
            X_te_lc[non_sens_cols].values,
            X_tr_lc[sens_col_name].values,
            X_te_lc[sens_col_name].values
        )

        bayes_aia_results.append({
            "Dataset": ds_name,
            "Seed": seed,
            "flat_bayes_aia_lift": flat_lift,
            "dpgbc_bayes_aia_lift": dpgbc_lift,
            "dplc_bayes_aia_lift": dplc_lift,
        })

# Per-dataset means
summary = pd.DataFrame(bayes_aia_results).groupby("Dataset")[
    ["flat_bayes_aia_lift", "dpgbc_bayes_aia_lift", "dplc_bayes_aia_lift"]
].mean().round(3)

print(summary.to_string())

In [ ]:
# Cell 28a (NEW): KD-tree-style DP baseline — reimplementation of the GENERAL
# RECIPE in Kreačić, Nouri, Potluru, Balch, Veloso, "Differentially Private
# Synthetic Data Using KD-Trees," UAI 2023, PMLR 216:1143-1153. This is NOT
# their verified official code — it is our best-effort reimplementation of
# the space-partitioning + noisy-count + box-sampling recipe described in
# their abstract, built by reusing our own already-verified DP primitives,
# for an internal, clearly-labeled comparison only. Do not report these
# numbers as a reproduction of their published results.
#
# Structural difference from DP-GBC (dp_generate_granular_balls_with_radius):
#   - DP-GBC splits are chosen by a LABEL-PURITY score (majority-count).
#   - This KD-tree baseline splits are chosen by a DENSITY-BALANCE score
#     (how evenly a threshold divides the node's point count), which does
#     not look at y at all -- consistent with KD-trees' general-purpose,
#     not classification-specific, design goal.
#   - Stopping is by count/depth only (no purity early-stop), since this
#     variant partitions the FEATURE SPACE, not label homogeneity.
# Same sensitivity-1 exponential-mechanism machinery as DP-GBC is reused,
# so the privacy accounting is identical in form.
#
# FIX (this revision): the ledger in run_dp_kdtree_pipeline now also bills
# the terminal-depth COUNT-ONLY release that dp_generate_kdtree_balls performs
# at every leaf at depth == max_depth. Previously it was omitted, so the
# pipeline reported total_eps == TOTAL_EPS while actually spending
# ≈ TOTAL_EPS + eps_count_per_level. Cell 28b's "matched ε=3.0" comparison
# is now fair in the strict accountant sense.

def balance_score(vals, t):
    """Sensitivity 1 everywhere: adding/removing one record shifts
    n_left or n_right by exactly 1, so the score changes by at most 1.
    Degenerate splits get a bounded penalty -(n_left + n_right) instead
    of a large negative constant, keeping the sensitivity bound tight."""
    left = vals <= t
    n_left, n_right = int(left.sum()), int((~left).sum())
    if n_left < 2 or n_right < 2:
        return -(n_left + n_right)
    return -abs(n_left - n_right)

def dp_generate_kdtree_balls(X, y, max_depth, eps_count_per_level,
                              eps_split_per_level, min_size=10,
                              n_candidate_thresholds=8, rng=None,
                              all_classes=None):
    rng = rng or np.random.default_rng(SEED)
    X_arr = X.values if hasattr(X, "values") else np.asarray(X)
    y_arr = np.asarray(y)
    global_classes = all_classes if all_classes is not None else np.unique(y_arr)
    n_features = X_arr.shape[1]
    balls = []
    root_lo, root_hi = np.zeros(n_features), np.ones(n_features)
    queue = [(np.arange(len(X_arr)), 0, root_lo, root_hi)]

    while queue:
        indices, depth, box_lo, box_hi = queue.pop(0)
        X_node, y_node = X_arr[indices], y_arr[indices]

        counts_full = np.array([np.sum(y_node == c) for c in global_classes])
        noisy_counts = np.array([
            laplace_mech(c, sensitivity=1.0, eps=eps_count_per_level, rng=rng)
            for c in counts_full
        ])
        noisy_counts = np.clip(noisy_counts, 0, None) + 1.0

        stop = (noisy_counts.sum() < min_size) or (depth >= max_depth)  # no purity check

        def make_leaf():
            center = (box_lo + box_hi) / 2.0
            radius = np.linalg.norm(box_hi - box_lo) / 2.0
            balls.append({
                "indices": indices, "center": center, "radius": radius,
                "box_lo": box_lo.copy(), "box_hi": box_hi.copy(),
                "n": len(indices), "classes": global_classes,
                "noisy_class_counts": np.clip(noisy_counts, 0, None)
            })

        if stop:
            make_leaf(); continue

        candidates, scores = [], []
        for f in range(n_features):
            lo_f, hi_f = box_lo[f], box_hi[f]
            if hi_f - lo_f < 1e-9:
                continue
            vals = X_node[:, f]
            thresholds = np.linspace(lo_f, hi_f, n_candidate_thresholds + 2)[1:-1]
            for t in thresholds:
                candidates.append((f, t))
                scores.append(balance_score(vals, t))

        if not candidates:
            make_leaf(); continue

        chosen_f, chosen_t = exponential_mechanism(candidates, scores, 1.0, eps_split_per_level, rng)
        left_mask = X_node[:, chosen_f] <= chosen_t

        left_lo, left_hi = box_lo.copy(), box_hi.copy()
        left_hi[chosen_f] = np.clip(min(left_hi[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])
        right_lo, right_hi = box_lo.copy(), box_hi.copy()
        right_lo[chosen_f] = np.clip(max(right_lo[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])

        queue.append((indices[left_mask], depth + 1, left_lo, left_hi))
        queue.append((indices[~left_mask], depth + 1, right_lo, right_hi))

    return balls


def run_dp_kdtree_pipeline(X, y, seed, lo, hi, max_depth=3,
                            eps_count_per_level=0.1, eps_split_per_level=0.1,
                            eps_count_release=2.4, all_classes=None,
                            precomputed_split=None, sampling_mode="uniform"):
    """Same budget-accounting shape as run_dp_gbc_pipeline, for a fair,
    matched-epsilon comparison. eps_total = (D+1)*eps_count + D*eps_split + eps_count_release,
    where D = max_depth: D internal split levels each charge a count + split
    release, and the terminal (depth==D) leaves each charge a count-only
    release."""
    rng = np.random.default_rng(seed)
    ledger = PrivacyLedger()

    if precomputed_split is not None:
        X_tr, X_te, y_tr, y_te = precomputed_split
    else:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)

    X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
    X_te_n = scale_with_public_bounds(X_te, lo, hi)

    balls = dp_generate_kdtree_balls(
        X_tr_n, y_tr.values, max_depth=max_depth,
        eps_count_per_level=eps_count_per_level,
        eps_split_per_level=eps_split_per_level,
        rng=rng, all_classes=all_classes
    )

    # --- Ledger accounting (matches actual builder spends) ---
    per_level = eps_count_per_level + eps_split_per_level
    for depth in range(max_depth):
        ledger.spend_sequential_charged_once(
            per_level, delta=0.0, n_branches=len(balls),
            note=f"kdtree granulation depth {depth+1} (count+split)"
        )
    # FIX: bill the terminal-depth count-only release that the builder performs.
    ledger.spend_sequential_charged_once(
        eps_count_per_level, delta=0.0, n_branches=len(balls),
        note="kdtree terminal depth (count only)"
    )
    ledger.spend_sequential(
        eps_count_release, 0.0, note="kdtree per-leaf synthetic-size release"
    )
    # ---------------------------------------------------------

    sizes = dp_release_ball_counts(balls, eps_count_release, rng)
    X_synth, y_synth = dp_sample_synthetic_records(
        balls, sizes, rng, d=X_tr_n.shape[1], sampling_mode=sampling_mode
    )
    return pd.DataFrame(X_synth, columns=X_tr_n.columns), X_te_n, pd.Series(y_synth), y_te, ledger, balls

In [ ]:
# Cell 28b (guarded) — KD-tree-style baseline on the 7 PRIMARY datasets.
# The run loop is unchanged; only the comparison block is wrapped in
# try/except NameError, because df_main lives in Cell 7 (which the 26-cell
# working set skips). The comparison is done in the new notebook after
# merging dpgbc_kdtree_baseline.csv + main_results.csv.

kdtree_results = []
_max_depth = 3
_eps_count = _eps_split = 0.1
_, _, _eps_count_release = allocate_budget_exact(
    TOTAL_EPS, _max_depth, _eps_count, _eps_split
)

for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets:
        continue
    X, y = datasets[ds_name]
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue
        fixed_split = (X_tr, X_te, y_tr, y_te)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)

        X_tr_kd, X_te_kd, y_tr_kd, y_te_kd, ledger_kd, balls_kd = run_dp_kdtree_pipeline(
            X, y, seed, lo, hi, max_depth=_max_depth,
            eps_count_per_level=_eps_count, eps_split_per_level=_eps_split,
            eps_count_release=_eps_count_release,
            all_classes=classes_public[ds_name],
            precomputed_split=fixed_split,
        )
        assert abs(ledger_kd.total_eps - TOTAL_EPS) < 1e-9

        if len(X_tr_kd) == 0:
            continue

        util_kd, model_kd = evaluate_utility(X_tr_kd, y_tr_kd, X_te_kd, y_te_kd)
        mia_kd, _ = membership_inference_attack_rf_synth(
            X_tr_kd, y_tr_kd, X_tr_n, y_tr, X_te_kd, y_te, seed
        )

        kdtree_results.append({
            "Dataset": ds_name, "Seed": seed,
            "kdtree_bal_acc": util_kd["balanced_accuracy"],
            "kdtree_mia_rf": mia_kd,
        })
    print(f"done kdtree-baseline: {ds_name}")

df_kdtree = pd.DataFrame(kdtree_results)
df_kdtree.to_csv("dpgbc_kdtree_baseline.csv", index=False)

# ---- Summary of what was written (always safe, no df_main) ----
print(f"\n[save] dpgbc_kdtree_baseline.csv  ({len(df_kdtree)} rows)")
print(f"       datasets: {sorted(df_kdtree.Dataset.unique())}")
print(f"       seeds per dataset: "
      f"{df_kdtree.groupby('Dataset').Seed.nunique().to_dict()}")

# ---- Comparison block: only if df_main is in scope ----
try:
    merged = df_kdtree.merge(
        df_main[["Dataset", "Seed", "flat_LR_bal_acc", "dplc_bal_acc", "dplc_MIA_rf"]],
        on=["Dataset", "Seed"],
    )
    comp_kdtree = merged.groupby("Dataset")[
        ["flat_LR_bal_acc", "kdtree_bal_acc", "dplc_bal_acc",
         "kdtree_mia_rf", "dplc_MIA_rf"]
    ].mean().round(3)
    print("\n=== DP-GBC vs. KD-tree-style baseline (matched epsilon=3.0) ===")
    print(comp_kdtree.to_string())

    from scipy.stats import wilcoxon as _wilcoxon2
    piv2 = merged.groupby("Dataset")[["kdtree_bal_acc", "dplc_bal_acc"]].mean().dropna()
    stat, p = _wilcoxon2(piv2["dplc_bal_acc"], piv2["kdtree_bal_acc"])
    print(f"\nWilcoxon, DP-GBC(+LR) vs. KD-tree utility, p={p:.5f}, "
          f"median diff = {np.median(piv2['dplc_bal_acc'] - piv2['kdtree_bal_acc']):+.4f}")
except NameError:
    print("\n[skip comparison] df_main not in scope — Cell 7 was skipped.")
    print("                  Wilcoxon is computed in the new notebook after")
    print("                  dpgbc_kdtree_baseline.csv is merged into df_main.")

In [ ]:
# ── (2) Leaf-count CV across your existing 5 seeds, exact Cell 7 config ──
leaf_counts = {ds: [] for ds in NUMERIC_DATASETS}
for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets: continue
    X, y = datasets[ds_name]
    lo, hi = public_bounds_for(ds_name, X)
    for seed in SEEDS:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        balls = dp_generate_granular_balls_with_radius(
            X_tr_n, y_tr.values, max_depth=3,
            eps_count_per_level=0.1, eps_split_per_level=0.1,
            eps_radius_per_level=0.0, eps_center_per_level=0.0,
            rng=np.random.default_rng(seed), all_classes=classes_public[ds_name])
        leaf_counts[ds_name].append(len(balls))

rows = []
for ds, c in leaf_counts.items():
    c = np.array(c, dtype=float)
    rows.append({"Dataset": ds, "mean_leaves": c.mean(),
                 "cv": c.std(ddof=1)/c.mean() if c.mean() > 0 else np.nan})
df_leafcv = pd.DataFrame(rows)
print(df_leafcv.sort_values("cv", ascending=False).to_string(index=False))
print(f"Median CV={df_leafcv['cv'].median():.3f}  Max CV={df_leafcv['cv'].max():.3f}")

In [ ]:
# ── (3) Per-leaf KS distance + sparse-bin oversampling, single-seed snapshot ──
# Note: use ONE rng per dataset, not a fresh default_rng(42) inside the loop
# (Qwen's version re-seeds every ball with the same seed — correlated noise, avoid it).
ks_rows, sparse_rows = [], []
for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets: continue
    X, y = datasets[ds_name]
    lo, hi = public_bounds_for(ds_name, X)
    seed = 42
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
    rng = np.random.default_rng(seed)          # one stream for this dataset
    balls = dp_generate_granular_balls_with_radius(
        X_tr_n, y_tr.values, max_depth=3,
        eps_count_per_level=0.1, eps_split_per_level=0.1,
        eps_radius_per_level=0.0, eps_center_per_level=0.0,
        rng=rng, all_classes=classes_public[ds_name])
    _, _, eps_count_release = allocate_budget_exact(3.0, 3, 0.1, 0.1)   # = 2.3
    sizes = dp_release_ball_counts(balls, eps_count_release, rng)

    ds_ks, ds_sparse = [], []
    for ball, n_synth in zip(balls, sizes):
        if n_synth <= 0: continue
        idx = ball["indices"]
        X_real_ball = X_tr_n.values[idx]
        box_lo, box_hi = ball["box_lo"], ball["box_hi"]
        X_synth_ball = rng.uniform(low=box_lo, high=box_hi, size=(n_synth, X_tr_n.shape[1]))
        for f in range(X_tr_n.shape[1]):
            real_f, synth_f = X_real_ball[:, f], X_synth_ball[:, f]
            if len(real_f) > 1 and np.ptp(real_f) > 1e-9:
                ds_ks.append(ks_2samp(real_f, synth_f)[0])
            bins = np.linspace(box_lo[f], box_hi[f], 11)
            hr, _ = np.histogram(real_f, bins=bins)
            hs, _ = np.histogram(synth_f, bins=bins)
            ds_sparse.append(((hr <= 1) & (hs > 0)).mean())
    if ds_ks: ks_rows.append({"Dataset": ds_name, "median_within_ball_ks": np.median(ds_ks)})
    if ds_sparse: sparse_rows.append({"Dataset": ds_name, "sparse_bin_rate": np.mean(ds_sparse)})

df_ks2, df_sp2 = pd.DataFrame(ks_rows), pd.DataFrame(sparse_rows)
print(df_ks2.to_string(index=False)); print(df_sp2.to_string(index=False))
print(f"Median within-ball KS={df_ks2['median_within_ball_ks'].median():.3f}")
print(f"Mean sparse-bin rate={df_sp2['sparse_bin_rate'].mean():.1%}  max={df_sp2['sparse_bin_rate'].max():.1%}")

In [ ]:

# Cell 29a: Tab-PE-style baseline (Private Evolution reimplementation)
# Simplified reimplementation of the Private Evolution mechanism Tab-PE builds on
# (Tran, Backurs, Lin, Reis, Xiong, Yekhanin, "Tab-PE," ICML 2026 / arXiv:2606.08259).
# NOT the authors' official code -- built from this notebook's own already-verified
# DP primitives (laplace_mech, PrivacyLedger, scale_with_public_bounds), following
# the same "clearly labeled, not the paper's code" convention as the KD-tree baseline
# in Cell 28a. Do not report these numbers as a reproduction of the paper's results.
#
#   1. Initialize a population of candidate synthetic rows (public, data-independent).
#   2. For each generation:
#        a. VARIATION (public post-processing): mutate/crossover candidates using
#           tabular-specialized operators (per-feature resample, two-point crossover).
#        b. PRIVATE SELECTION: each real private row casts one vote for its nearest
#           candidate (L2, on public-bounds-scaled features). The vote histogram has
#           L2 sensitivity sqrt(2) under add/remove (one person's vote moves from one
#           bin to another), released via Gaussian mechanism -- same sigma formula as
#           make_flat_noised (Cell 7) for consistency.
#        c. Keep the top-`population_size` candidates by noisy vote count.
#   3. Budget: eps_total split across n_generations (85%) + a final label release (15%).
#
# Labels are attached post-hoc via DP-released per-candidate class-count histograms
# (same Laplace mechanism as dp_release_ball_counts), no extra generations needed.

def _tabpe_variation(parents, rng, lo=0.0, hi=1.0, mutation_rate=0.3):
    """Tabular-specialized operators: per-feature resample + crossover.
    Public post-processing -- no privacy cost."""
    n, d = parents.shape
    children = parents.copy()
    mut_mask = rng.random((n, d)) < mutation_rate
    children[mut_mask] = rng.uniform(lo, hi, size=mut_mask.sum())
    partner_idx = rng.permutation(n)
    cut = rng.integers(1, d, size=n)
    for i in range(n):
        if rng.random() < 0.5:
            j = partner_idx[i]
            c = cut[i]
            children[i, c:] = parents[j, c:]
    return children


def run_tabpe_pipeline(X, y, seed, lo, hi, eps_total, n_generations=8,
                        population_size=None, delta=1e-5, all_classes=None,
                        precomputed_split=None):
    rng = np.random.default_rng(seed)
    ledger = PrivacyLedger()

    if precomputed_split is not None:
        X_tr, X_te, y_tr, y_te = precomputed_split
    else:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y
        )

    X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
    X_te_n = scale_with_public_bounds(X_te, lo, hi)
    X_arr = X_tr_n.values
    d = X_arr.shape[1]
    n_real = len(X_arr)

    population_size = population_size or min(500, max(50, n_real // 4))
    eps_per_gen = eps_total * 0.85 / n_generations
    eps_label_release = eps_total * 0.15

    population = rng.uniform(0.0, 1.0, size=(population_size, d))

    for gen in range(n_generations):
        children = _tabpe_variation(population, rng)
        candidates = np.vstack([population, children])
        n_cand = len(candidates)

        dists = cdist(X_arr, candidates)
        votes = np.argmin(dists, axis=1)
        hist = np.bincount(votes, minlength=n_cand).astype(float)

        sigma = np.sqrt(2.0) * np.sqrt(2 * np.log(1.25 / delta)) / max(eps_per_gen, 1e-6)
        noisy_hist = hist + rng.normal(0, sigma, size=n_cand)

        ledger.spend_sequential(eps_per_gen, delta, note=f"tabpe generation {gen+1} vote release")

        top_idx = np.argsort(noisy_hist)[::-1][:population_size]
        population = candidates[top_idx]

    dists_final = cdist(X_arr, population)
    nearest = np.argmin(dists_final, axis=1)
    global_classes = all_classes if all_classes is not None else np.unique(y_tr.values)
    y_tr_arr = y_tr.values
    class_counts = np.zeros((population_size, len(global_classes)))
    for i, c in enumerate(global_classes):
        mask = y_tr_arr == c
        counts_per_cand = np.bincount(nearest[mask], minlength=population_size)
        noisy_counts = np.array([
            laplace_mech(v, sensitivity=1.0, eps=eps_label_release / len(global_classes), rng=rng)
            for v in counts_per_cand
        ])
        class_counts[:, i] = np.clip(noisy_counts, 0, None)
    ledger.spend_sequential(eps_label_release, 0.0, note="tabpe per-candidate label release (pure-eps)")

    totals = class_counts.sum(axis=1)
    y_synth = np.array([
        global_classes[np.argmax(class_counts[i])] if totals[i] > 0 else global_classes[0]
        for i in range(population_size)
    ])

    X_tr_out = pd.DataFrame(population, columns=X_tr_n.columns)
    y_tr_out = pd.Series(y_synth)
    return X_tr_out, X_te_n, y_tr_out, y_te, ledger

In [ ]:
# Cell 29b: Run Tab-PE-style baseline, matched epsilon/datasets/seeds as Cell 28b

tabpe_results = []
N_GENERATIONS = 8

for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets:
        continue
    X, y = datasets[ds_name]
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue
        fixed_split = (X_tr, X_te, y_tr, y_te)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)

        X_tr_tp, X_te_tp, y_tr_tp, y_te_tp, ledger_tp = run_tabpe_pipeline(
            X, y, seed, lo, hi, eps_total=TOTAL_EPS, n_generations=N_GENERATIONS,
            all_classes=classes_public[ds_name], precomputed_split=fixed_split
        )
        assert abs(ledger_tp.total_eps - TOTAL_EPS) < 1e-6

        if len(X_tr_tp) == 0 or y_tr_tp.nunique() < 2:
            continue

        util_tp, model_tp = evaluate_utility(X_tr_tp, y_tr_tp, X_te_tp, y_te)
        mia_tp, _ = membership_inference_attack_rf_synth(
            X_tr_tp, y_tr_tp, X_tr_n, y_tr, X_te_tp, y_te, seed
        )

        tabpe_results.append({
            "Dataset": ds_name, "Seed": seed,
            "tabpe_bal_acc": util_tp["balanced_accuracy"],
            "tabpe_mia_rf": mia_tp,
        })
    print(f"done tabpe-baseline: {ds_name}")

df_tabpe = pd.DataFrame(tabpe_results)
df_tabpe.to_csv("dpgbc_tabpe_baseline.csv", index=False)

# ---- Summary of what was written (always safe, no df_main) ----
print(f"\n[save] dpgbc_tabpe_baseline.csv  ({len(df_tabpe)} rows)")
print(f"       datasets: {sorted(df_tabpe.Dataset.unique())}")
print(f"       seeds per dataset: "
      f"{df_tabpe.groupby('Dataset').Seed.nunique().to_dict()}")

# ---- Comparison block: only if df_main is in scope ----
try:
    merged_tp = df_tabpe.merge(
        df_main[["Dataset", "Seed", "flat_LR_bal_acc", "dplc_bal_acc", "dplc_MIA_rf"]],
        on=["Dataset", "Seed"],
    )
    comp_tabpe = merged_tp.groupby("Dataset")[
        ["flat_LR_bal_acc", "tabpe_bal_acc", "dplc_bal_acc",
         "tabpe_mia_rf", "dplc_MIA_rf"]
    ].mean().round(3)
    print("\n=== DP-GBC vs. Tab-PE-style baseline (matched epsilon=3.0) ===")
    print(comp_tabpe.to_string())

    from scipy.stats import wilcoxon
    piv_tp = merged_tp.groupby("Dataset")[["tabpe_bal_acc", "dplc_bal_acc"]].mean().dropna()
    stat, p = wilcoxon(piv_tp["dplc_bal_acc"], piv_tp["tabpe_bal_acc"])
    print(f"\nWilcoxon, DP-GBC(+LR) vs. Tab-PE utility, p={p:.5f}, "
          f"median diff = {np.median(piv_tp['dplc_bal_acc'] - piv_tp['tabpe_bal_acc']):+.4f}")
except NameError:
    print("\n[skip comparison] df_main not in scope — Cell 7 was skipped.")
    print("                  Wilcoxon is computed in the new notebook after")
    print("                  dpgbc_tabpe_baseline.csv is merged into df_main.")